# P1 Day 1: Data Pipeline for Assignment IV

This notebook prepares a model-ready log-return dataset for the S&P 500 replication task.

Key rules used here:
- Train window: January 2020 to December 2024
- Validation window: January 2025 to June 2025
- Final test window: July 2025 to December 2025
- Empty CSVs and non-price CSVs are excluded and reported
- Late entrants and early exits are excluded from the final return matrix rather than shrinking the window into 2025
- A one-month grace period is allowed at the start of training; any stock starting after that is excluded
- Slight validation/test holes are forward-filled at the price level before return construction


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.options.display.max_rows = 200
pd.options.display.max_columns = 20

DATA_DIR = Path("Mega")
OUTPUT_DIR = Path(".")
INDEX_FILE = DATA_DIR / "^GSPC.csv"
CACHE_FILE = DATA_DIR / "sp500_tickers_cache.csv"

TRAIN_START = pd.Timestamp("2020-01-01")
TRAIN_PRICE_GRACE_END = pd.Timestamp("2020-02-03")
TRAIN_END = pd.Timestamp("2024-12-31")
VAL_START = pd.Timestamp("2025-01-01")
VAL_END = pd.Timestamp("2025-06-30")
TEST_START = pd.Timestamp("2025-07-01")
TEST_END = pd.Timestamp("2025-12-31")
FINAL_WINDOW_END = TEST_END

print("Data directory:", DATA_DIR.resolve())
print("Output directory:", OUTPUT_DIR.resolve())


Data directory: /Users/tanmaygawande/Desktop/Code/Sem2/AI in Finance/DA6701-Data-Science-and-AI-in-Finance/Assignment4/Mega
Output directory: /Users/tanmaygawande/Desktop/Code/Sem2/AI in Finance/DA6701-Data-Science-and-AI-in-Finance/Assignment4


In [2]:
def load_close_series(path: Path):
    ticker = path.stem

    try:
        raw = pd.read_csv(path)
    except Exception as exc:
        issue = {
            "ticker": ticker,
            "file": path.name,
            "reason": "read_error",
            "details": f"{type(exc).__name__}: {exc}",
        }
        return None, issue

    if raw.empty:
        issue = {
            "ticker": ticker,
            "file": path.name,
            "reason": "empty_dataframe",
            "details": "CSV has no rows.",
        }
        return None, issue

    required_columns = {"Date", "Close"}
    if not required_columns.issubset(raw.columns):
        issue = {
            "ticker": ticker,
            "file": path.name,
            "reason": "invalid_schema",
            "details": ", ".join(raw.columns),
        }
        return None, issue

    cleaned = raw.loc[:, ["Date", "Close"]].copy()
    cleaned["Date"] = pd.to_datetime(cleaned["Date"], errors="coerce")
    cleaned["Close"] = pd.to_numeric(cleaned["Close"], errors="coerce")
    cleaned = cleaned.dropna(subset=["Date", "Close"])
    cleaned = cleaned.sort_values("Date").drop_duplicates(subset="Date", keep="last")

    if cleaned.empty:
        issue = {
            "ticker": ticker,
            "file": path.name,
            "reason": "no_valid_prices",
            "details": "No usable Date/Close rows remained after cleaning.",
        }
        return None, issue

    close_series = cleaned.set_index("Date")["Close"].rename(ticker)

    metadata = {
        "ticker": ticker,
        "file": path.name,
        "first_date": close_series.index.min(),
        "last_date": close_series.index.max(),
        "row_count": int(close_series.shape[0]),
    }
    return close_series, metadata


def assign_coverage_reason(first_date: pd.Timestamp, last_date: pd.Timestamp) -> str:
    late_entry = first_date > TRAIN_PRICE_GRACE_END
    early_exit = last_date < FINAL_WINDOW_END

    if late_entry and early_exit:
        return "late_entry_and_early_exit"
    if late_entry:
        return "late_entry"
    if early_exit:
        return "early_exit"
    return "kept"


def missing_by_split(price_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "train_missing": price_df.loc[:TRAIN_END].isna().sum(),
            "val_missing": price_df.loc[VAL_START:VAL_END].isna().sum(),
            "test_missing": price_df.loc[TEST_START:TEST_END].isna().sum(),
        }
    ).sort_index()


In [3]:
benchmark_close, benchmark_result = load_close_series(INDEX_FILE)
if benchmark_close is None:
    raise ValueError(f"Benchmark file could not be loaded: {benchmark_result}")

stock_series = {}
stock_metadata = []
file_issues = []

for csv_path in sorted(DATA_DIR.glob("*.csv")):
    if csv_path.name == INDEX_FILE.name:
        continue
    if csv_path.name == CACHE_FILE.name:
        file_issues.append(
            {
                "ticker": csv_path.stem,
                "file": csv_path.name,
                "reason": "not_price_history",
                "details": "Cache file, not a stock or index OHLCV history.",
            }
        )
        continue

    close_series, result = load_close_series(csv_path)
    if close_series is None:
        file_issues.append(result)
        continue

    stock_series[result["ticker"]] = close_series
    stock_metadata.append(result)

stock_meta_df = pd.DataFrame(stock_metadata).sort_values("ticker").reset_index(drop=True)
file_issues_df = pd.DataFrame(file_issues).sort_values(["reason", "file"]).reset_index(drop=True)

coverage_mask = (
    stock_meta_df["first_date"].le(TRAIN_PRICE_GRACE_END)
    & stock_meta_df["last_date"].ge(FINAL_WINDOW_END)
)

kept_meta_df = stock_meta_df.loc[coverage_mask].copy().reset_index(drop=True)
excluded_meta_df = stock_meta_df.loc[~coverage_mask].copy().reset_index(drop=True)
excluded_meta_df["reason"] = excluded_meta_df.apply(
    lambda row: assign_coverage_reason(row["first_date"], row["last_date"]),
    axis=1,
)

benchmark_calendar = benchmark_close.loc[TRAIN_START:FINAL_WINDOW_END].index
candidate_prices = pd.DataFrame(
    {ticker: stock_series[ticker].reindex(benchmark_calendar) for ticker in kept_meta_df["ticker"]},
    index=benchmark_calendar,
)

first_valid_dates = candidate_prices.apply(pd.Series.first_valid_index)
common_price_start = first_valid_dates.max()

if common_price_start > TRAIN_PRICE_GRACE_END:
    raise ValueError(
        "The common training start drifted beyond the allowed grace period. "
        f"Observed start: {common_price_start.date()}"
    )

candidate_prices = candidate_prices.loc[common_price_start:FINAL_WINDOW_END].sort_index()
train_missing_raw = candidate_prices.loc[:TRAIN_END].isna().sum()
train_complete_tickers = train_missing_raw[train_missing_raw.eq(0)].index.tolist()
train_gap_exclusions_df = (
    train_missing_raw[train_missing_raw.gt(0)]
    .rename("train_missing_points")
    .reset_index()
    .rename(columns={"index": "ticker"})
    .sort_values(["train_missing_points", "ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

final_prices_raw = candidate_prices.loc[:, train_complete_tickers].copy()
missing_before_fill = missing_by_split(final_prices_raw)
final_prices = final_prices_raw.ffill()
missing_after_fill = missing_by_split(final_prices)

if missing_after_fill["train_missing"].sum() != 0:
    raise ValueError("Training prices still contain holes after universe selection.")
if missing_after_fill["val_missing"].sum() != 0:
    raise ValueError("Validation prices still contain holes after forward fill.")
if missing_after_fill["test_missing"].sum() != 0:
    raise ValueError("Test prices still contain holes after forward fill.")

benchmark_prices = benchmark_close.reindex(final_prices.index).ffill()
stock_returns = np.log(final_prices / final_prices.shift(1)).dropna(how="any")
benchmark_returns = np.log(benchmark_prices / benchmark_prices.shift(1)).dropna().rename("^GSPC")

common_return_index = stock_returns.index.intersection(benchmark_returns.index)
stock_returns = stock_returns.loc[common_return_index]
benchmark_returns = benchmark_returns.loc[common_return_index]
model_ready_returns = stock_returns.join(benchmark_returns, how="inner")

train_returns = model_ready_returns.loc[:TRAIN_END].copy()
val_returns = model_ready_returns.loc[VAL_START:VAL_END].copy()
test_returns = model_ready_returns.loc[TEST_START:TEST_END].copy()

summary_df = pd.DataFrame(
    [
        {"metric": "benchmark_rows_in_window", "value": int(len(benchmark_calendar))},
        {"metric": "readable_stock_files", "value": int(len(stock_meta_df))},
        {"metric": "issue_file_count", "value": int(len(file_issues_df))},
        {"metric": "coverage_kept_stock_count", "value": int(len(kept_meta_df))},
        {"metric": "coverage_excluded_stock_count", "value": int(len(excluded_meta_df))},
        {"metric": "train_gap_excluded_stock_count", "value": int(len(train_gap_exclusions_df))},
        {"metric": "final_stock_count", "value": int(stock_returns.shape[1])},
        {"metric": "common_price_start", "value": common_price_start.strftime("%Y-%m-%d")},
        {"metric": "final_return_start", "value": stock_returns.index.min().strftime("%Y-%m-%d")},
        {"metric": "final_return_end", "value": stock_returns.index.max().strftime("%Y-%m-%d")},
        {"metric": "train_rows", "value": int(train_returns.shape[0])},
        {"metric": "val_rows", "value": int(val_returns.shape[0])},
        {"metric": "test_rows", "value": int(test_returns.shape[0])},
    ]
)

print("Benchmark window:", benchmark_calendar.min().date(), "to", benchmark_calendar.max().date())
print("Common training price start:", common_price_start.date())
print("Final stock return matrix shape:", stock_returns.shape)
print("Train / Val / Test shapes:", train_returns.shape, val_returns.shape, test_returns.shape)

display(summary_df)
display(file_issues_df)
display(excluded_meta_df.head(25))
display(train_gap_exclusions_df.head(25))
display(missing_before_fill.sum().to_frame("missing_points_before_fill"))
display(missing_after_fill.sum().to_frame("missing_points_after_fill"))


Benchmark window: 2020-01-02 to 2025-12-31
Common training price start: 2020-01-02
Final stock return matrix shape: (1507, 553)
Train / Val / Test shapes: (1257, 554) (122, 554) (128, 554)


,metric,value
0,benchmark_rows_in_window,1508
1,readable_stock_files,578
2,issue_file_count,6
3,coverage_kept_stock_count,554
4,coverage_excluded_stock_count,24
5,train_gap_excluded_stock_count,1
6,final_stock_count,553
7,common_price_start,2020-01-02
8,final_return_start,2020-01-03
9,final_return_end,2025-12-31


,ticker,file,reason,details
0,ANDV,ANDV.csv,empty_dataframe,CSV has no rows.
1,ESRX,ESRX.csv,empty_dataframe,CSV has no rows.
2,EVHC,EVHC.csv,empty_dataframe,CSV has no rows.
3,NFX,NFX.csv,empty_dataframe,CSV has no rows.
4,SCG,SCG.csv,empty_dataframe,CSV has no rows.
5,sp500_tickers_cache,sp500_tickers_cache.csv,not_price_history,"Cache file, not a stock or index OHLCV history."


,ticker,file,first_date,last_date,row_count,reason
0,ABNB,ABNB.csv,2020-12-10,2026-01-27,1287,late_entry
1,APP,APP.csv,2021-04-15,2026-01-27,1202,late_entry
2,CARR,CARR.csv,2020-03-19,2026-01-27,1472,late_entry
3,CEG,CEG.csv,2022-01-19,2026-01-27,1009,late_entry
4,COIN,COIN.csv,2021-04-14,2026-01-27,1203,late_entry
5,COL,COL.csv,2020-01-02,2020-11-30,228,early_exit
6,DASH,DASH.csv,2020-12-09,2026-01-27,1288,late_entry
7,DDR,DDR.csv,2020-01-02,2021-09-30,423,early_exit
8,EXE,EXE.csv,2021-02-10,2026-01-27,1246,late_entry
9,FB,FB.csv,2025-06-26,2026-01-27,148,late_entry


,ticker,train_missing_points
0,CVG,25


,missing_points_before_fill
train_missing,0
val_missing,0
test_missing,1


,missing_points_after_fill
train_missing,0
val_missing,0
test_missing,0


In [4]:
artifacts = {
    "p1d1_stock_returns.parquet": stock_returns,
    "p1d1_benchmark_returns.parquet": benchmark_returns.to_frame(),
    "p1d1_model_ready_returns.parquet": model_ready_returns,
    "p1d1_train_returns.parquet": train_returns,
    "p1d1_val_returns.parquet": val_returns,
    "p1d1_test_returns.parquet": test_returns,
    "p1d1_final_prices.parquet": final_prices,
    "p1d1_kept_universe.parquet": kept_meta_df,
    "p1d1_excluded_universe.parquet": excluded_meta_df,
}

for file_name, frame in artifacts.items():
    frame.to_parquet(OUTPUT_DIR / file_name, index=True)

file_issues_df.to_csv(OUTPUT_DIR / "p1d1_file_issues.csv", index=False)
train_gap_exclusions_df.to_csv(OUTPUT_DIR / "p1d1_train_gap_exclusions.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "p1d1_summary.csv", index=False)

saved_files = sorted(
    [
        path.name
        for path in OUTPUT_DIR.glob("p1d1_*")
        if path.is_file()
    ]
)

print("Saved artifacts:")
for name in saved_files:
    print("-", name)


Saved artifacts:
- p1d1_benchmark_returns.parquet
- p1d1_excluded_universe.parquet
- p1d1_file_issues.csv
- p1d1_final_prices.parquet
- p1d1_kept_universe.parquet
- p1d1_model_ready_returns.parquet
- p1d1_stock_returns.parquet
- p1d1_summary.csv
- p1d1_test_returns.parquet
- p1d1_train_gap_exclusions.csv
- p1d1_train_returns.parquet
- p1d1_val_returns.parquet
